In [12]:
import random
import operator
import math
import numpy as np
from deap import base, creator, tools

In [13]:
# Number of variables
NUM_DECISION_VARIABLES = 20

# Population size and number of generations
POPULATION_SIZE = 50
NUM_OF_GEN = 1000

# Search space limits
POS_MIN = -30
POS_MAX = 30

# Note: Please change here the evalutation function name evaluation_rosenbrock / evaluation_griewank
EVALUATION_FUNC_NAME = "evaluation_griewank"

EVALUATION_FUNCTIONS = {
    "evaluation_rosenbrock": evaluation_rosenbrock,
    "evaluation_griewank": evaluation_griewank
}

toolbox.register(
    "evaluate",
    EVALUATION_FUNCTIONS[EVALUATION_FUNC_NAME]
)

# Meta-EP settings
META_EP_BETA = (POS_MAX - POS_MIN) / 10
META_EP_C = 1
META_EP_EPSILON = META_EP_BETA / 10

In [14]:
# Make a random individual with a variance for each variable.
def generate_individual(size, pos_min, pos_max):
    individual = creator.Individual(random.uniform(pos_min, pos_max) for _ in range(size))
    individual.variance = [random.gauss(0, 1) for _ in range(len(individual))]

    # Do not let the variance get too small.
    individual.variance = [
        max(v, META_EP_EPSILON)
        for v in individual.variance
    ]
    return individual


def mutate_meta_ep(current_individual):
    # Random values used to mutate the solution and variance.
    rx = [random.gauss(0, 1) for _ in range(len(current_individual))]
    rv = [random.gauss(0, 1) for _ in range(len(current_individual))]

    # Change the solution.
    new_individual = creator.Individual(
        list(
            map(
                operator.add,
                current_individual,
                map(
                    operator.mul,
                    rx,
                    map(math.sqrt, current_individual.variance)
                )
            )
        )
    )

    # Change the variance used for the next mutation.
    new_individual.variance = list(
        map(
            operator.add,
            current_individual.variance,
            map(
                operator.mul,
                rv,
                map(
                    math.sqrt,
                    (META_EP_C * v for v in current_individual.variance)
                )
            )
        )
    )

    # Keep the variance above epsilon.
    new_individual.variance = [
        max(v, META_EP_EPSILON)
        for v in new_individual.variance
    ]

    return new_individual


# Fitness functions
def evaluation_rosenbrock(individual):
    fitness = sum(100 * (individual[i] ** 2 - individual[i + 1]) ** 2 + (individual[i] - 1) ** 2 for i in range(len(individual) - 1))
    return (fitness,)


def evaluation_griewank(individual):
    sum_part = sum(x ** 2 / 4000 for x in individual)
    product_part = math.prod(math.cos(x / math.sqrt(i + 1)) for i, x in enumerate(individual))

    fitness = sum_part - product_part + 1

    return (fitness,)

In [15]:
# Set up the fitness and individual classes.
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin, variance=list)

# Set up the toolbox.
toolbox = base.Toolbox()

toolbox.register("individual",
                 generate_individual,
                 size=NUM_DECISION_VARIABLES,
                 pos_min=POS_MIN,
                 pos_max=POS_MAX)

# Create the population.
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("mutate", mutate_meta_ep)

# Use the Griewank function for fitness.
toolbox.register("evaluate", EVALUATION_FUNCTIONS[EVALUATION_FUNC_NAME])

# Keep the best individuals.
toolbox.register("select", tools.selBest, k=POPULATION_SIZE)

In [16]:
# Fix the seed so that the evolutionary experiment is reproducible.
seed_list = list(range(1, 31))
results = []

for seed in seed_list:
    random.seed(seed)
    
    initial_population = toolbox.population(POPULATION_SIZE)
    
    current_population = initial_population
    
    # Evaluate the population before starting evolutionary selection.
    for individual in current_population:
        individual.fitness.values = toolbox.evaluate(individual)
    
    for gen in range(NUM_OF_GEN):
        new_population = []
        for individual in current_population:
            # Create one offspring by mutating the parent's solution and variances.
            new_mutated_individual = toolbox.mutate(individual)
            new_population.append(new_mutated_individual)
    
        # Evaluate the newly generated offspring.
        for individual in new_population:
            individual.fitness.values = toolbox.evaluate(individual)
    
        # Combine parents and offspring for elitist survival selection.
        total_population = current_population + new_population
    
        # Retain the best N individuals for the next generation.
        current_population = toolbox.select(total_population)

    best_individual = toolbox.select(current_population, k = 1)[0]
    results.append({
        "seed": seed,
        "best_solution": list(best_individual),
        "fitness": best_individual.fitness.values[0]
    })
    print(f'Seed : {seed}, function: {EVALUATION_FUNC_NAME}, Fitness: {best_individual.fitness.values[0]}')

Seed : 1, function: evaluation_griewank, Fitness: 0.5521903098064103
Seed : 2, function: evaluation_griewank, Fitness: 0.6442770214338824
Seed : 3, function: evaluation_griewank, Fitness: 0.5190081881642233
Seed : 4, function: evaluation_griewank, Fitness: 0.45490163419717333
Seed : 5, function: evaluation_griewank, Fitness: 0.6528741696227138
Seed : 6, function: evaluation_griewank, Fitness: 0.6709040776267092
Seed : 7, function: evaluation_griewank, Fitness: 0.5723359414426777
Seed : 8, function: evaluation_griewank, Fitness: 0.6265115259909242
Seed : 9, function: evaluation_griewank, Fitness: 0.4525988036379357
Seed : 10, function: evaluation_griewank, Fitness: 0.680918444401305
Seed : 11, function: evaluation_griewank, Fitness: 0.520368129129531
Seed : 12, function: evaluation_griewank, Fitness: 0.5306718304773745
Seed : 13, function: evaluation_griewank, Fitness: 0.46529504279527
Seed : 14, function: evaluation_griewank, Fitness: 0.5292487291115413
Seed : 15, function: evaluation_

In [20]:
fitness_values = [result["fitness"] for result in results]

mean_fitness = np.mean(fitness_values)
std_fitness = np.std(fitness_values, ddof=1)

print("Mean:", mean_fitness)
print("Std:", std_fitness)

Mean: 0.5716034554772033
Std: 0.07894834175662642
